# MNIST Hardware Neural Network — Training & Weight Export

This notebook trains a neural network that **exactly mirrors** the SystemVerilog hardware design:

```
Input (28×28 uint8)
  └─► 2×2 Average Pooling  → 14×14 = 196 values  (int8, >>2)
        └─► Dense 196→32   → 32 values            (int32 acc, ReLU, >>6 → int16)
              └─► Dense 32→10  → 10 values         (int32 acc, ReLU, >>6 → int16)
                    └─► ArgMax → digit (0–9)
```

All weights/biases are quantized to **int8** to match the hardware registers.
The notebook also runs a **bit-exact simulation** of the hardware arithmetic so you can verify outputs before flashing the FPGA.

---

## 1. Install & Import Dependencies

In [ ]:
# Install if needed
# !pip install tensorflow numpy matplotlib

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Hardware Parameters
These constants mirror the SystemVerilog parameters exactly.

In [ ]:
# ── Hardware architecture constants ───────────────────────────────────────────
IMG_SIZE      = 28          # input image side length
POOL_SIZE     = 2           # avg_pooling kernel size (2×2)
POOL_OUT      = 14          # 28/2 = 14
POOL_FLAT     = POOL_OUT ** 2  # 196 — IN_SIZE for dense_layer1

L1_NEURONS    = 32          # dense_layer1 output width
L2_NEURONS    = 10          # dense_layer2 output width (one per digit)

WIDTH         = 8           # weight / bias bit-width
W_MIN         = -(2**(WIDTH-1))      # -128
W_MAX         =  (2**(WIDTH-1)) - 1  # +127

RELU_SHIFT    = 6           # hardware: out = relu(acc) >>> 6

print(f"Pool output size  : {POOL_OUT}×{POOL_OUT} = {POOL_FLAT} values")
print(f"Weight range      : [{W_MIN}, {W_MAX}]")
print(f"ReLU shift        : >>> {RELU_SHIFT}  (÷ {2**RELU_SHIFT})")

## 3. Load & Preprocess MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"Train samples : {x_train.shape[0]}")
print(f"Test  samples : {x_test.shape[0]}")
print(f"Pixel range   : [{x_train.min()}, {x_train.max()}]")

In [ ]:
# ── Hardware avg-pooling: integer arithmetic, shift-right by 2 ────────────────
# Matches: pool_out <= (in1 + in2 + in3 + in4) >>> 2
def hw_avg_pool(images_uint8: np.ndarray) -> np.ndarray:
    """
    Input : (N, 28, 28) uint8   [0, 255]
    Output: (N, 196)    int8    — bit-exact to avg_pooling.sv
    """
    N = images_uint8.shape[0]
    imgs = images_uint8.astype(np.int32)          # promote to avoid overflow
    # Reshape to (N, 14, 2, 14, 2) then sum the 2×2 block
    blocks = imgs.reshape(N, POOL_OUT, POOL_SIZE, POOL_OUT, POOL_SIZE)
    summed = blocks.sum(axis=(2, 4))              # (N, 14, 14), int32
    # Arithmetic right-shift by 2 (matches >>> 2 for positive values here)
    pooled = summed >> 2                           # (N, 14, 14), effectively ÷4
    pooled = pooled.reshape(N, POOL_FLAT)         # (N, 196)
    # The hardware stores this as signed 8-bit (the sum can be at most 4×255=1020
    # >>> 2 = 255 which fits in uint8; clamp to int8 range just in case)
    return np.clip(pooled, W_MIN, W_MAX).astype(np.int8)


x_train_pool = hw_avg_pool(x_train)   # (60000, 196)
x_test_pool  = hw_avg_pool(x_test)    # (10000, 196)

print(f"After pooling train : {x_train_pool.shape}  dtype={x_train_pool.dtype}")
print(f"After pooling test  : {x_test_pool.shape}   dtype={x_test_pool.dtype}")
print(f"Pooled value range  : [{x_train_pool.min()}, {x_train_pool.max()}]")

In [ ]:
# Visualise one example to confirm pooling looks correct
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(x_train[0], cmap='gray')
axes[0].set_title(f'Original 28×28  label={y_train[0]}')
axes[0].axis('off')
axes[1].imshow(x_train_pool[0].reshape(POOL_OUT, POOL_OUT), cmap='gray')
axes[1].set_title('After 2×2 avg-pool → 14×14')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Build & Train the Floating-Point Model

We train in **float32** first, then quantise weights to int8 to match the hardware.  
The architecture is kept identical to the RTL:
- Input: 196 (pooled, flattened)
- Dense 196 → 32, ReLU
- Dense 32 → 10, ReLU
- Softmax for training (hardware uses argmax directly)

In [ ]:
# Normalise to [0, 1] for stable gradient flow during training
x_train_f = x_train_pool.astype(np.float32) / 127.0
x_test_f  = x_test_pool.astype(np.float32)  / 127.0

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test,  10)

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential([
    keras.layers.Input(shape=(POOL_FLAT,)),
    # Layer 2 in hardware: dense_layer1 (196 → 32)
    keras.layers.Dense(L1_NEURONS, activation='relu',
                       kernel_initializer='glorot_uniform',
                       bias_initializer='zeros',
                       name='dense_layer1'),
    # Layer 3 in hardware: dense_layer2 (32 → 10)
    keras.layers.Dense(L2_NEURONS, activation='relu',
                       kernel_initializer='glorot_uniform',
                       bias_initializer='zeros',
                       name='dense_layer2'),
    # Softmax only for training — hardware uses argmax
    keras.layers.Softmax(name='softmax_output')
], name='hardware_nn')

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True,
                                  monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3,
                                      monitor='val_loss', verbose=1)
]

history = model.fit(
    x_train_f, y_train_cat,
    epochs=40,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'],     label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['accuracy'],     label='train acc')
axes[1].plot(history.history['val_accuracy'], label='val acc')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

_, float_acc = model.evaluate(x_test_f, y_test_cat, verbose=0)
print(f"\nFloat model test accuracy: {float_acc*100:.2f}%")

## 5. Quantise Weights to int8

The hardware uses `signed [7:0]` (int8, range −128…127) for weights and biases.  
We scale then clip-and-round the float weights to fit.

> **Tip:** If accuracy drops too much after quantisation, try training with a smaller learning rate for more epochs so weights stay small, or reduce the scale factor.

In [ ]:
def quantise_to_int8(arr: np.ndarray, scale: float = 127.0) -> np.ndarray:
    """
    Scale float weights → int8, matching hardware signed [7:0].
    scale: multiply factor before rounding.
    Clip to [-128, 127].
    """
    scaled  = arr * scale
    rounded = np.round(scaled).astype(np.int32)
    clipped = np.clip(rounded, W_MIN, W_MAX)
    return clipped.astype(np.int8)


def get_layer_weights(layer_name: str, w_scale: float = 127.0, b_scale: float = 1.0):
    """Return (weights_int8, biases_int8) for a named Dense layer."""
    layer   = model.get_layer(layer_name)
    w_float, b_float = layer.get_weights()
    w_int8  = quantise_to_int8(w_float, scale=w_scale)
    b_int8  = quantise_to_int8(b_float, scale=b_scale)
    return w_int8, b_int8


# ── Quantise ─────────────────────────────────────────────────────────────────
# Weights: scale by 127 to fill the int8 dynamic range
# Biases : scale by 1 (they're already small; adjust if needed)
W1, B1 = get_layer_weights('dense_layer1', w_scale=127.0, b_scale=1.0)
W2, B2 = get_layer_weights('dense_layer2', w_scale=127.0, b_scale=1.0)

print("dense_layer1 weights shape:", W1.shape, "  dtype:", W1.dtype)
print("dense_layer1 biases  shape:", B1.shape, "  dtype:", B1.dtype)
print("dense_layer2 weights shape:", W2.shape, "  dtype:", W2.dtype)
print("dense_layer2 biases  shape:", B2.shape, "  dtype:", B2.dtype)

print(f"\nW1 range: [{W1.min()}, {W1.max()}]")
print(f"B1 range: [{B1.min()}, {B1.max()}]")
print(f"W2 range: [{W2.min()}, {W2.max()}]")
print(f"B2 range: [{B2.min()}, {B2.max()}]")

## 6. Bit-Exact Hardware Simulation

Simulate the **exact integer arithmetic** from the SystemVerilog, including:
- `int8` inputs × `int8` weights → `int32` accumulator
- Bias add (int8 → sign-extended, added to int32)
- `relu(acc) >>> RELU_SHIFT` → int16 output (matches `relu.sv`)

In [ ]:
def hw_relu_shift(acc: np.ndarray, shift: int = RELU_SHIFT) -> np.ndarray:
    """
    Matches relu.sv:
        temp = (data_in > 0) ? data_in : 0;
        data_out = temp >>> 6;
    acc  : int32 array
    out  : int16 array  (truncated to lower 16 bits like SV does)
    """
    relu_val = np.where(acc > 0, acc, np.int32(0))      # ReLU
    shifted  = relu_val >> shift                          # arithmetic right-shift
    # Hardware takes [15:0] of the 32-bit shifted value
    return np.clip(shifted, -32768, 32767).astype(np.int16)


def hw_dense(in_data: np.ndarray,
             weights: np.ndarray,
             biases:  np.ndarray) -> np.ndarray:
    """
    Bit-exact dense layer.
    in_data : (N, IN_SIZE)          int16  (pooled layer) or int16 (after relu)
    weights : (IN_SIZE, NEURONS)    int8
    biases  : (NEURONS,)            int8
    returns : (N, NEURONS)          int32  (accumulator, before relu)
    
    Matches neuron.sv:
        product_w = sign_extend(in_data[addr]) * sign_extend(weight[addr])
        out += product_w  ... neuron_out = out + bias
    """
    # Sign-extend both operands to int32 before multiply (matches SV sign-extension)
    x = in_data.astype(np.int32)   # (N, IN_SIZE)
    w = weights.astype(np.int32)   # (IN_SIZE, NEURONS)
    b = biases.astype(np.int32)    # (NEURONS,)
    acc = x @ w + b                # (N, NEURONS), int32
    return acc


def hw_forward(images_uint8: np.ndarray) -> np.ndarray:
    """
    Full hardware-accurate forward pass.
    images_uint8 : (N, 28, 28) uint8
    returns      : (N,)        int   predicted digits
    """
    # Step 1: avg_pooling_layer
    pooled   = hw_avg_pool(images_uint8)           # (N, 196) int8

    # Step 2: dense_layer1 → relu (W1 shape: (196, 32), note Keras stores (in, out))
    acc1     = hw_dense(pooled.astype(np.int16), W1, B1)   # (N, 32) int32
    out1     = hw_relu_shift(acc1)                          # (N, 32) int16

    # Step 3: dense_layer2 → relu
    acc2     = hw_dense(out1, W2, B2)                       # (N, 10) int32
    out2     = hw_relu_shift(acc2)                          # (N, 10) int16

    # Step 4: select_max (argmax)
    return np.argmax(out2, axis=1)


# ── Evaluate hw simulation accuracy ──────────────────────────────────────────
hw_preds  = hw_forward(x_test)
hw_acc    = np.mean(hw_preds == y_test)
print(f"Hardware-simulation test accuracy: {hw_acc*100:.2f}%")
print(f"Float model test accuracy        : {float_acc*100:.2f}%")
print(f"Accuracy drop from quantisation  : {(float_acc - hw_acc)*100:.2f}%")

## 7. Export Weights as SystemVerilog `localparam`

Generates the `localparam` arrays you paste directly into `dense_layer1.sv` and `dense_layer2.sv`.

In [ ]:
def array_to_sv_localparam(arr: np.ndarray, name: str, dtype_str: str = "signed [7:0]") -> str:
    """
    Convert a 1-D or 2-D int8 numpy array to a SystemVerilog localparam string.
    For 2-D: arr shape is (NEURONS, IN_SIZE) — rows = one neuron's weight vector.
    Keras stores weights as (IN_SIZE, NEURONS), so we transpose first.
    """
    lines = []
    if arr.ndim == 1:
        # Bias vector (NEURONS,)
        n = arr.shape[0]
        vals = ", ".join(str(int(v)) for v in arr)
        lines.append(f"localparam {dtype_str} {name} [0:{n-1}] = '{{ {vals} }};")

    elif arr.ndim == 2:
        # Weight matrix: arr is (IN_SIZE, NEURONS) from Keras → transpose to (NEURONS, IN_SIZE)
        mat  = arr.T   # (NEURONS, IN_SIZE)
        n_neurons, in_size = mat.shape
        lines.append(f"localparam {dtype_str} {name} [0:{n_neurons-1}][0:{in_size-1}] = '{{")
        for i, row in enumerate(mat):
            vals    = ", ".join(str(int(v)) for v in row)
            comma   = "," if i < n_neurons - 1 else ""
            lines.append(f"    {{ {vals} }}{comma}")
        lines.append("};")

    return "\n".join(lines)


sv_b1 = array_to_sv_localparam(B1, "B_ARRAY_L2")
sv_w1 = array_to_sv_localparam(W1, "W_ARRAY_L2")
sv_b2 = array_to_sv_localparam(B2, "B_ARRAY_L3")
sv_w2 = array_to_sv_localparam(W2, "W_ARRAY_L3")

print("=== dense_layer1.sv  (paste into module body) ===")
print(sv_b1[:200], "...")
print()
print("=== dense_layer2.sv  (paste into module body) ===")
print(sv_b2)
print(sv_w2[:300], "...")

In [ ]:
# Save to files
with open("weights_dense_layer1.sv", "w") as f:
    f.write("// dense_layer1.sv  — drop-in localparam replacements\n")
    f.write("// Generated by mnist_hardware_nn.ipynb\n\n")
    f.write(sv_b1 + "\n\n")
    f.write(sv_w1 + "\n")

with open("weights_dense_layer2.sv", "w") as f:
    f.write("// dense_layer2.sv  — drop-in localparam replacements\n")
    f.write("// Generated by mnist_hardware_nn.ipynb\n\n")
    f.write(sv_b2 + "\n\n")
    f.write(sv_w2 + "\n")

print("Saved: weights_dense_layer1.sv")
print("Saved: weights_dense_layer2.sv")

## 8. Export Test Vectors for the Testbench

Generates the `logic [7:0] test_data_X` arrays used in `NN_tb.sv`, so you can compare the hardware output against the Python-predicted label.

In [ ]:
def image_to_sv_array(img_2d: np.ndarray, var_name: str) -> str:
    """
    Convert a (28, 28) uint8 image to an SV test vector.
    Matches the format in NN_tb.sv.
    Note: pixel values are cast to signed int8 for SV (0-127 safe; 128-255 wrap to neg).
    """
    flat = img_2d.flatten().astype(np.uint8)
    # Reinterpret >127 as their int8 signed equivalent (e.g. 255 → -1)
    signed_flat = flat.view(np.int8)
    vals = ", ".join(str(int(v)) for v in signed_flat)
    return f"logic signed [7:0] {var_name} [0:783] = '{{ {vals} }};"


# Pick one sample per digit class for a balanced test set
samples = {}
for idx in range(len(x_test)):
    label = y_test[idx]
    if label not in samples:
        samples[label] = idx
    if len(samples) == 10:
        break

print("=== NN_tb.sv test vectors ===\n")
tb_lines = []
for digit in range(10):
    idx   = samples[digit]
    img   = x_test[idx]
    label = y_test[idx]
    hw_pred = hw_forward(img[np.newaxis])[0]
    line = image_to_sv_array(img, f"test_digit_{digit}")
    tb_lines.append(line)
    print(f"Digit {digit}  (sample idx {idx:5d})  HW-sim predicts: {hw_pred}  "
          f"{'✓' if hw_pred == digit else '✗ MISMATCH'}")

with open("test_vectors.sv", "w") as f:
    f.write("// test_vectors.sv — generated by mnist_hardware_nn.ipynb\n")
    f.write("// Paste these into NN_tb.sv to replace test_data_a/b/c\n\n")
    f.write("\n".join(tb_lines) + "\n")

print("\nSaved: test_vectors.sv")

## 9. Visualise the Test Digit Predictions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for digit in range(10):
    idx     = samples[digit]
    img     = x_test[idx]
    hw_pred = hw_forward(img[np.newaxis])[0]

    axes[digit].imshow(img, cmap='gray')
    color = 'green' if hw_pred == digit else 'red'
    axes[digit].set_title(f'True: {digit}  HW: {hw_pred}', color=color, fontsize=10)
    axes[digit].axis('off')

plt.suptitle('Hardware-Simulation Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Per-Digit Confidence Scores (Layer 2 Output)

Useful for debugging: shows the raw `out2` (int16) values before argmax,
which is what `dense2_res` holds in the RTL.

In [ ]:
def hw_forward_verbose(image_2d: np.ndarray):
    """Return intermediate values for a single image — useful for RTL debugging."""
    img_batch = image_2d[np.newaxis]                         # (1, 28, 28)

    pooled = hw_avg_pool(img_batch)                          # (1, 196) int8
    acc1   = hw_dense(pooled.astype(np.int16), W1, B1)      # (1, 32)  int32
    out1   = hw_relu_shift(acc1)                             # (1, 32)  int16
    acc2   = hw_dense(out1, W2, B2)                         # (1, 10)  int32
    out2   = hw_relu_shift(acc2)                             # (1, 10)  int16
    pred   = int(np.argmax(out2, axis=1)[0])

    return {
        'pooled_img':  pooled[0],      # 196 values — matches AvgPoolingLayer.pool
        'dense1_acc':  acc1[0],        # 32 int32 — raw accumulator before relu
        'dense1_out':  out1[0],        # 32 int16 — matches dense1_res in top
        'dense2_acc':  acc2[0],        # 10 int32
        'dense2_out':  out2[0],        # 10 int16 — matches dense2_res in top
        'prediction':  pred
    }


# Show debug output for digit '7'
result = hw_forward_verbose(x_test[samples[7]])
print(f"Predicted digit  : {result['prediction']}")
print(f"\nPooled image (first 20 of 196): {result['pooled_img'][:20]}")
print(f"\nDense-1 output (int16, 32 values):\n{result['dense1_out']}")
print(f"\nDense-2 raw acc (int32, 10 values — compare with dense2_res in sim):\n{result['dense2_acc']}")
print(f"\nDense-2 output (int16, 10 values — compare with dense2_res after relu):\n{result['dense2_out']}")

# Bar chart of final scores
plt.figure(figsize=(8, 3))
plt.bar(range(10), result['dense2_out'], color=['green' if i == result['prediction'] else 'steelblue' for i in range(10)])
plt.xticks(range(10))
plt.xlabel('Digit class')
plt.ylabel('Score (int16)')
plt.title(f"Layer-2 output scores  →  predicted digit: {result['prediction']}")
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 11. Export Weights as NumPy Arrays (for scripting)

In [ ]:
np.save("W1_int8.npy", W1)   # shape (196, 32)
np.save("B1_int8.npy", B1)   # shape (32,)
np.save("W2_int8.npy", W2)   # shape (32, 10)
np.save("B2_int8.npy", B2)   # shape (10,)

print("Saved: W1_int8.npy, B1_int8.npy, W2_int8.npy, B2_int8.npy")
print("\nTo reload:")
print("  W1 = np.load('W1_int8.npy')")
print("  B1 = np.load('B1_int8.npy')")

## 12. Summary & How to Use the Outputs

| File | Description | Use |
|------|-------------|-----|
| `weights_dense_layer1.sv` | `B_ARRAY_L2` + `W_ARRAY_L2` localparam | Replace hardcoded params in `dense_layer1.sv` |
| `weights_dense_layer2.sv` | `B_ARRAY_L3` + `W_ARRAY_L3` localparam | Replace hardcoded params in `dense_layer2.sv` |
| `test_vectors.sv` | 10 test images (one per digit) as SV arrays | Replace `test_data_a/b/c` in `NN_tb.sv` |
| `W1_int8.npy` etc. | NumPy int8 arrays | Further Python experiments |

### Workflow

1. **Run this notebook** → trains the model, quantises weights, runs HW simulation.
2. **Copy** `B_ARRAY_L2`/`W_ARRAY_L2` from `weights_dense_layer1.sv` into `dense_layer1.sv`.
3. **Copy** `B_ARRAY_L3`/`W_ARRAY_L3` from `weights_dense_layer2.sv` into `dense_layer2.sv`.
4. **Copy** a test vector from `test_vectors.sv` into `NN_tb.sv`, set `img = test_digit_X;`.
5. **Run simulation** and compare `digit_out` against the Python-predicted label above.

### Improving Accuracy

- If hardware accuracy < float accuracy by more than ~3%, try **smaller weight scale** (e.g. 64 instead of 127) so fewer values saturate at ±127.
- Train with **weight regularisation** (`kernel_regularizer=keras.regularizers.l2(1e-4)`) to keep weights small and friendly to int8.
- The `RELU_SHIFT = 6` (÷64) can cause underflow for small activations — increasing `L1_NEURONS` from 32 to 64 gives more headroom.